[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adaptionlabs/adaption-runbooks/blob/main/adaptive-data/combine_datasets.ipynb)

# Combine Datasets

Validate that several completed Adaptive Data datasets are compatible, then merge them into one new dataset. Sources are left untouched.

**Docs:** [Combine datasets](https://docs.adaptionlabs.ai/adaptive-data/combine-datasets/)  
**Prerequisites:** an Adaption API key ([create one](https://adaptionlabs.ai/app/settings?tab=api_keys)) set as `ADAPTION_API_KEY` (env var or Colab secret).

> ⏱ Runtime: minutes · 💳 Cost: low — combining is cheap; building the sample sources below costs a few credits

In [ ]:
%pip install -q adaption

## Setup

Load your API key. In Colab, add `ADAPTION_API_KEY` to the secrets panel (🔑 in the sidebar) and enable it for this notebook. Running locally, the cell prompts for the key if the environment variable is not set.

In [ ]:
import os

from adaption import Adaption

if "ADAPTION_API_KEY" not in os.environ:
    try:
        from google.colab import userdata  # Colab secrets

        os.environ["ADAPTION_API_KEY"] = userdata.get("ADAPTION_API_KEY")
    except ImportError:
        from getpass import getpass

        os.environ["ADAPTION_API_KEY"] = getpass("ADAPTION_API_KEY: ")

client = Adaption()

## Parameters

Each source must belong to your organization and have completed an Adaptive Data run. If you already have two such datasets (for example from the [Configure Adaptive Data](https://docs.adaptionlabs.ai/adaptive-data/configure-adaptive-data/) runbook), set `CREATE_SOURCES = False` and list their IDs. Otherwise the next step builds two small ones for you.

In [ ]:
CREATE_SOURCES = True
SOURCE_DATASET_IDS = [
    # "dataset_abc123",
    # "dataset_def456",
]
COMBINED_NAME = "Combined runbook dataset"

## Step 1 — Get two source datasets

When `CREATE_SOURCES` is true, this helper imports a small slice of two public Hugging Face datasets and runs a tiny Adaptive Data pass on each (`max_rows=60`) so both are in a combinable state. Skip this if you supplied your own IDs.

In [ ]:
import time


def make_adapted_dataset(url, files, column_mapping, max_rows=60):
    ds = client.datasets.create(source={"url": url, "files": files})
    dataset_id = ds.dataset_id

    while True:
        status = client.datasets.get_status(dataset_id)
        if status.status == "failed":
            raise RuntimeError(f"Ingestion failed: {status.error_data}")
        if status.row_count is not None:
            break
        time.sleep(5)

    client.datasets.adapt(
        dataset_id,
        column_mapping=column_mapping,
        job_specification={"max_rows": max_rows},
    )
    status = client.datasets.wait_for_completion(dataset_id, timeout=3600)
    if status.status == "failed":
        raise RuntimeError(f"Adaptation failed: {status.error_data}")
    return dataset_id


if CREATE_SOURCES:
    SOURCE_DATASET_IDS = [
        make_adapted_dataset(
            "https://huggingface.co/datasets/tatsu-lab/alpaca",
            ["data/train-00000-of-00001-a09b74b3ef9c3b56.parquet"],
            {"prompt": "instruction", "completion": "output", "context": ["input"]},
        ),
        make_adapted_dataset(
            "https://huggingface.co/datasets/openai/gsm8k",
            ["main/train-00000-of-00001.parquet"],
            {"prompt": "question", "completion": "answer"},
        ),
    ]

print(SOURCE_DATASET_IDS)

## Step 2 — Validate before combining

Validation checks compatibility without writing data. **Errors** block the merge (a source not ready, owned by another org, or mixing text and multimodal data). **Warnings** describe reconciliation the merge performs automatically, such as casting mismatched column types or resolving mixed training types.

In [ ]:
validation = client.datasets.combine.validate(dataset_ids=SOURCE_DATASET_IDS)

for issue in validation.issues:
    print(issue.level, issue.code, issue.message)

if not validation.compatible:
    raise ValueError("The datasets cannot be combined")

print("Resolved training type:", validation.training_type)
print("Estimated total rows:", validation.total_estimated_rows)

## Step 3 — Combine

`combine.create` runs synchronously and returns the new `dataset_id`. The `idempotency_key` makes retries safe: a repeat with the same key returns the dataset created by the first request instead of merging again.

In [ ]:
from uuid import uuid4

combined = client.datasets.combine.create(
    dataset_ids=SOURCE_DATASET_IDS,
    name=COMBINED_NAME,
    idempotency_key=str(uuid4()),
)
combined_id = combined.dataset_id
print("Combined dataset:", combined_id)

## Step 4 — Inspect the combined dataset

Datasets with the same training type keep it; combining instruction and preference datasets produces an instruction dataset.

In [ ]:
dataset = client.datasets.get(dataset_id=combined_id)
print("Name:  ", dataset.name)
print("Rows:  ", dataset.row_count)
print("Status:", dataset.status)

## Next steps

- [Train a model with AutoScientist](https://docs.adaptionlabs.ai/autoscientist-quickstart/) on the combined dataset.
- [Download](https://docs.adaptionlabs.ai/adaptive-data-quickstart/#download-the-adapted-dataset) it for your own pipeline.

## Clean up

Combining created one new dataset; the sources are unchanged. Uncomment to delete the merged result.

In [ ]:
# client.datasets.delete(dataset_id=combined_id)